In [39]:
from nhlpy import NHLClient
import pandas as pd
import polars as pl

In [2]:
client = NHLClient(
    debug=True,           # Enable debug logging
    timeout=30,           # Request timeout in seconds
    ssl_verify=True,      # SSL certificate verification
    follow_redirects=True # Follow HTTP redirects
)

teams = client.teams.teams()
standings = client.standings.league_standings()
games = client.schedule.daily_schedule()

nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/standings/now
nhlpy.http_client - DEBUG - GET: https://api.nhle.com/stats/rest/en/franchise
nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/standings/now
nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/schedule/2026-05-22


In [14]:
client.teams.team_roster(team_abbr='PHI', season=20062007)

nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/roster/PHI/20062007


{'forwards': [{'id': 8467397,
   'headshot': 'https://assets.nhle.com/mugs/nhl/20062007/PHI/8467397.png',
   'firstName': {'default': 'Dimitri'},
   'lastName': {'default': 'Afanasenkov'},
   'sweaterNumber': 27,
   'positionCode': 'L',
   'shootsCatches': 'R',
   'heightInInches': 74,
   'weightInPounds': 209,
   'heightInCentimeters': 188,
   'weightInKilograms': 95,
   'birthDate': '1980-05-12',
   'birthCity': {'default': 'Arkhangelsk',
    'cs': 'Archandělsk',
    'fi': 'Arkangeli',
    'sk': 'Archandělsk',
    'sv': 'Arkangelsk'},
   'birthCountry': 'RUS'},
  {'id': 8466260,
   'headshot': 'https://assets.nhle.com/mugs/nhl/20062007/PHI/8466260.png',
   'firstName': {'default': 'Kyle'},
   'lastName': {'default': 'Calder'},
   'sweaterNumber': 19,
   'positionCode': 'L',
   'shootsCatches': 'L',
   'heightInInches': 71,
   'weightInPounds': 177,
   'heightInCentimeters': 180,
   'weightInKilograms': 80,
   'birthDate': '1979-01-05',
   'birthCity': {'default': 'Mannville'},
   'bi

In [ ]:
game_example = client.schedule.daily_schedule(date='2025-11-10')['games']
cols = ['id', 'season', 'startTimeUTC', 'gameType', 'venue', 'homeTeam', 'awayTeam', 'periodDescriptor', 'gameOutcome']
games = [{k:v for k,v in game.items() if k in cols} for game in game_example]
games = pd.json_normalize(games, sep='_')

nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/schedule/2025-11-10


,id,season,gameType,startTimeUTC,venue_default,awayTeam_id,awayTeam_commonName_default,awayTeam_placeName_default,awayTeam_placeNameWithPreposition_default,awayTeam_placeNameWithPreposition_fr,...,homeTeam_abbrev,homeTeam_logo,homeTeam_darkLogo,homeTeam_homeSplitSquad,homeTeam_score,periodDescriptor_number,periodDescriptor_periodType,periodDescriptor_maxRegulationPeriods,gameOutcome_lastPeriodType,awayTeam_placeName_fr
0,2025020252,20252026,2,2025-11-11T00:00:00Z,Prudential Center,2,Islanders,New York,New York,de New York,...,NJD,https://assets.nhle.com/logos/nhl/svg/NJD_ligh...,https://assets.nhle.com/logos/nhl/svg/NJD_dark...,False,2,4,OT,3,OT,NaN
1,2025020253,20252026,2,2025-11-11T00:00:00Z,Madison Square Garden,18,Predators,Nashville,Nashville,de Nashville,...,NYR,https://assets.nhle.com/logos/nhl/svg/NYR_ligh...,https://assets.nhle.com/logos/nhl/svg/NYR_dark...,False,6,3,REG,3,REG,NaN
2,2025020254,20252026,2,2025-11-11T01:30:00Z,Rogers Place,29,Blue Jackets,Columbus,Columbus,de Columbus,...,EDM,https://assets.nhle.com/logos/nhl/svg/EDM_ligh...,https://assets.nhle.com/logos/nhl/svg/EDM_dark...,False,5,4,OT,3,OT,NaN
3,2025020255,20252026,2,2025-11-11T03:00:00Z,T-Mobile Arena,13,Panthers,Florida,Florida,de la Floride,...,VGK,https://assets.nhle.com/logos/nhl/svg/VGK_ligh...,https://assets.nhle.com/logos/nhl/svg/VGK_dark...,False,2,3,REG,3,REG,Floride


In [43]:
from typing import Literal

def get_daily_game_table(
        date: str,
        client: NHLClient,
        mode: Literal['pandas', 'polars'] = 'pandas'
    ) -> None|pd.DataFrame|pl.DataFrame:
        
    games = client.schedule.daily_schedule(date=date)['games']
    games_cleaned = [{k:v for k,v in game.items() if k in cols} for game in games]

    if mode == 'pandas' or mode == 'csv':
        result = pd.json_normalize(games_cleaned, sep='_')
        result['startTimeUTC'] = pd.to_datetime(result.startTimeUTC)
        result = result.set_index('id')
    elif mode == 'polars':
        result = pl.json_normalize(games_cleaned, separator='_')
        result = result.with_columns(
            pl.col("startTimeUTC").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ").alias("startTimeUTC")
        )
    else:
        raise ValueError(f'Unrecognized mode: {mode}')

    return result

In [59]:
from typing import Literal
from datetime import datetime, timedelta
import pandas as pd
import polars as pl
from tqdm.auto import tqdm

def get_games_in_range(
    start_date: str,
    end_date: str,
    client,
    mode: Literal['pandas', 'polars', 'csv'] = 'pandas'
) -> None | pd.DataFrame | pl.DataFrame:
    """
    Fetch all games between start_date and end_date (inclusive), show a tqdm progress bar
    and display the last completed date in the bar. Dates must be 'YYYY-MM-DD'.
    """
    # Validate date format YYYY-MM-DD
    try:
        start_dt = datetime.strptime(start_date, "%Y-%m-%d").date()
        end_dt = datetime.strptime(end_date, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError("Invalid date format. Please use YYYY-MM-DD.")

    if end_dt < start_dt:
        raise ValueError("end_date must be the same or after start_date")

    # Normalize mode
    if mode == 'csv':
        mode = 'pandas'
    if mode not in ('pandas', 'polars'):
        raise ValueError(f"Unrecognized mode: {mode}")

    # Build inclusive list of dates
    dates = []
    cur = start_dt
    while cur <= end_dt:
        dates.append(cur.strftime("%Y-%m-%d"))
        cur += timedelta(days=1)

    pandas_frames = []
    polars_frames = []

    # tqdm progress bar: show last completed date via postfix
    with tqdm(dates, desc="Fetching games", unit="day") as bar:
        for d in bar:
            # fetch daily; get_daily_game_table returns None for empty days
            try:
                daily = get_daily_game_table(date=d, client=client, mode=mode)
            except Exception:
                # still update bar to show this date as completed (failed)
                bar.set_postfix(last_completed=d)
                continue

            # update postfix to show last completed date (successful or empty)
            bar.set_postfix(last_completed=d)

            if daily is None:
                continue

            if mode == 'pandas':
                if isinstance(daily, pd.DataFrame):
                    if daily.index.name == 'id' and 'id' not in daily.columns:
                        daily = daily.reset_index()
                    pandas_frames.append(daily)
                else:
                    # unexpected type, skip
                    continue
            else:  # polars
                if isinstance(daily, pl.DataFrame):
                    polars_frames.append(daily)
                else:
                    continue

    # No data found
    if mode == 'pandas' and not pandas_frames:
        return None
    if mode == 'polars' and not polars_frames:
        return None

    # Concatenate and deduplicate
    if mode == 'pandas':
        result = pd.concat(pandas_frames, ignore_index=True, sort=False)
        if 'id' not in result.columns and result.index.name == 'id':
            result = result.reset_index()
        if 'id' in result.columns:
            result = result.drop_duplicates(subset=['id'], keep='first').set_index('id')
        if 'startTimeUTC' in result.columns:
            result['startTimeUTC'] = pd.to_datetime(result['startTimeUTC'], errors='coerce')
        return result

    else:  # polars
        result = pl.concat(polars_frames, how='vertical')
        if 'id' in result.columns:
            result = result.unique(subset=['id'])
        if 'startTimeUTC' in result.columns:
            result = result.with_columns(
                pl.col("startTimeUTC").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ").alias("startTimeUTC")
            )
        return result


c:\Users\david\OneDrive\Desktop\nhl-analytics\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [60]:
get_games_in_range('2025-10-01', '2026-04-30', NHLClient())

Fetching games: 100%|██████████| 212/212 [01:45<00:00,  2.01day/s, last_completed=2026-04-30]


,season,gameType,neutralSite,startTimeUTC,easternUTCOffset,venueUTCOffset,venueTimezone,gameState,gameScheduleState,tvBroadcasts,...,seriesStatus_seriesAbbrev,seriesStatus_seriesTitle,seriesStatus_seriesLetter,seriesStatus_neededToWin,seriesStatus_topSeedTeamAbbrev,seriesStatus_topSeedWins,seriesStatus_bottomSeedTeamAbbrev,seriesStatus_bottomSeedWins,seriesStatus_gameNumberOfSeries,periodDescriptor_otPeriods
id,,,,,,,,,,,,,,,,,,,,,
2025010076,20252026,1,False,2025-10-01 23:00:00+00:00,-04:00,-04:00,America/New_York,FINAL,OK,"[{'id': 528, 'market': 'A', 'countryCode': 'US...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025010074,20252026,1,False,2025-10-02 01:00:00+00:00,-04:00,-07:00,US/Pacific,FINAL,OK,"[{'id': 324, 'market': 'N', 'countryCode': 'US...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025010077,20252026,1,False,2025-10-02 01:00:00+00:00,-04:00,-06:00,US/Mountain,FINAL,OK,"[{'id': 283, 'market': 'N', 'countryCode': 'CA...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025010073,20252026,1,False,2025-10-02 02:00:00+00:00,-04:00,-07:00,US/Pacific,FINAL,OK,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025010075,20252026,1,False,2025-10-02 02:00:00+00:00,-04:00,-07:00,US/Pacific,FINAL,OK,"[{'id': 554, 'market': 'H', 'countryCode': 'US...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025030125,20252026,3,False,2026-04-29 23:00:00+00:00,-04:00,-04:00,US/Eastern,OFF,OK,"[{'id': 307, 'market': 'N', 'countryCode': 'US...",...,R1,1st Round,B,4.0,TBL,2.0,MTL,3.0,5.0,NaN
2025030146,20252026,3,False,2026-04-29 23:30:00+00:00,-04:00,-04:00,US/Eastern,OFF,OK,"[{'id': 385, 'market': 'N', 'countryCode': 'US...",...,R1,1st Round,D,4.0,PIT,2.0,PHI,4.0,6.0,NaN
2025030175,20252026,3,False,2026-04-30 02:00:00+00:00,-04:00,-07:00,US/Pacific,OFF,OK,"[{'id': 385, 'market': 'N', 'countryCode': 'US...",...,R1,1st Round,G,4.0,VGK,3.0,UTA,2.0,5.0,2.0


In [56]:
game_example = client.schedule.daily_schedule(date='2025-10-05')['games']
game_example

nhlpy.http_client - DEBUG - GET: https://api-web.nhle.com/v1/schedule/2025-10-05


[]